In [4]:
# Sel ini menghasilkan TIGA dataset cabang (kota) terpisah untuk Tugas Mandiri Pertemuan 3
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)  # seed berbeda tiap kota agar datanya bervariasi, namun tetap konsisten/reproducible
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [11]:
!hdfs dfs -mkdir -p /user/raamxh/ecommerce/raw
!hdfs dfs -mkdir -p /user/raamxh/ecommerce/processed
!hdfs dfs -ls /user/raamxh/ecommerce

Found 2 items
drwxr-xr-x   - raamxh supergroup          0 2026-09-08 19:15 /user/raamxh/ecommerce/processed
drwxr-xr-x   - raamxh supergroup          0 2026-09-08 19:03 /user/raamxh/ecommerce/raw


In [5]:
!hdfs dfs -put transaksi_magelang.csv /user/raamxh/ecommerce/raw/
!hdfs dfs -put transaksi_yogyakarta.csv /user/raamxh/ecommerce/raw/
!hdfs dfs -put transaksi_semarang.csv /user/raamxh/ecommerce/raw/

!hdfs dfs -ls -h /user/raamxh/ecommerce/raw

Found 3 items
-rw-r--r--   1 raamxh supergroup     12.0 K 2026-09-08 19:02 /user/raamxh/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 raamxh supergroup     11.9 K 2026-09-08 19:03 /user/raamxh/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 raamxh supergroup     12.4 K 2026-09-08 19:02 /user/raamxh/ecommerce/raw/transaksi_yogyakarta.csv


In [6]:
import os
import subprocess
import pandas as pd

os.environ['CLASSPATH'] = subprocess.check_output(['hadoop', 'classpath', '--glob']).decode('utf-8').strip()
df_mgl = pd.read_csv("hdfs://localhost:9000/user/raamxh/ecommerce/raw/transaksi_magelang.csv")
df_ygy = pd.read_csv("hdfs://localhost:9000/user/raamxh/ecommerce/raw/transaksi_yogyakarta.csv")
df_smg = pd.read_csv("hdfs://localhost:9000/user/raamxh/ecommerce/raw/transaksi_semarang.csv")

df_gabungan = pd.concat([df_mgl, df_ygy, df_smg], ignore_index=True)
print(df_gabungan['kota'].value_counts())

kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64


In [ ]:
!pip install fsspec pyarrow

In [9]:
df_gabungan['total_pendapatan'] = df_gabungan['unit_terjual'] * df_gabungan['harga_satuan']
df_ringkasan = df_gabungan.groupby(['kota', 'kategori',])['total_pendapatan'].sum().reset_index()

df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
df_ringkasan.to_csv("ringkasan_kota_kategori.csv", index=False)

!hdfs dfs -put data_gabungan_bersih.csv /user/raamxh/ecommerce/processed/
!hdfs dfs -put ringkasan_kota_kategori.csv /user/raamxh/ecommerce/processed/

!hdfs dfs -ls /user/raamxh/ecommerce/processed

Found 2 items
-rw-r--r--   1 raamxh supergroup      41257 2026-09-08 19:15 /user/raamxh/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 raamxh supergroup        530 2026-09-08 19:15 /user/raamxh/ecommerce/processed/ringkasan_kota_kategori.csv


#Screenshot UI NameNode

![Screenshot from 2026-09-08 19-26-54.png](attachment:5935a125-2590-4bc0-adcd-2cee18debb1b.png)


*Apa keuntungan menyimpan data mentah (raw) terpisah dari data olahan (processed) di HDFS, dibandingkan menyimpan semuanya bercampur dalam satu folder?*
Memisahkan data mentah (raw) dan data olahan (processed) di dalam HDFS adalah sebuah langkah yang sangat penting untuk menjaga keamanan dan keaslian data. Mengapa demikian? Bayangkan saja jika kita langsung mengubah data asli tanpa menyimpannya terlebih dahulu. Jika suatu saat kita melakukan kesalahan saat menghitung rumus, salah mengetik kode, atau komputer mengalami error di tengah jalan, seluruh data asli kita bisa rusak atau bahkan hilang selamanya. Tentu ini akan sangat merepotkan. Dengan membuat dua folder yang terpisah, kita menjadikan folder raw sebagai brankas penyimpanan untuk data asli yang sama sekali tidak boleh disentuh atau diubah. Sementara itu, folder processed digunakan khusus untuk menampung hasil kerja atau hitungan kita. Keuntungannya, jika hasil olahan kita di folder processed ternyata salah, kita tidak perlu panik. Kita cukup menghapus data yang salah tersebut, lalu mengambil salinan data yang masih utuh dari folder raw, dan memulai kembali dari awal. Cara ini membuat pekerjaan kita jauh lebih aman.